# Part 2: Feature Engineering & Summary


## 1. Load Cleaned Dataset

In [1]:
import pandas as pd
import numpy as np

# Load the cleaned data saved in Part 1
df = pd.read_csv('titanic_cleaned.csv')
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Feature Engineering


In [2]:
# Feature 1: FamilySize
# Combining SibSp (Siblings/Spouses) and Parch (Parents/Children) plus 1 (themselves)
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Feature 2: IsAlone
# Binary feature: 1 if traveling alone (FamilySize == 1), else 0
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Feature 3: HasCabin
# Binary feature: 1 if they have a cabin number, 0 if it was missing (NaN)
df['HasCabin'] = df['Cabin'].notnull().astype(int)
# Now drop the original Cabin column as it has too many missing values
df = df.drop('Cabin', axis=1)

# Feature 4: Title
# Extract title from the Name column (e.g., Mr., Mrs., Miss.)
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
# Group rare titles into 'Rare'
rare_titles = ['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
df['Title'] = df['Title'].replace(rare_titles, 'Rare')
df['Title'] = df['Title'].replace('Mlle', 'Miss')
df['Title'] = df['Title'].replace('Ms', 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')

print("New Features Engineered successfully!")
df[['Name', 'Title', 'FamilySize', 'IsAlone', 'HasCabin']].head()

New Features Engineered successfully!


,Name,Title,FamilySize,IsAlone,HasCabin
0,"Braund, Mr. Owen Harris",Mr,2,0,0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs,2,0,1
2,"Heikkinen, Miss. Laina",Miss,1,1,0
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs,2,0,1
4,"Allen, Mr. William Henry",Mr,1,1,0


## 3. Feature Scaling (Standardization)
Earlier in EDA, we checked the distributions for `Age` (roughly normal but with some skew) and `Fare` (highly right-skewed, which we capped at the 99th percentile).


To fix this, we apply **Standard Scaling**, which transforms numerical features to have a mean of 0 and a standard deviation of 1. This puts all numerical features on the exact same scale without losing their underlying distribution.

In [3]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Select continuous numerical columns to scale
cols_to_scale = ['Age', 'Fare']

# Apply scaling and replace the original columns with the scaled versions
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

print("Scaling Applied Successfully!")
df[['Age', 'Fare']].head()

Scaling Applied Successfully!


,Age,Fare
0,-0.565736,-0.564109
1,0.663861,0.942548
2,-0.258337,-0.548227
3,0.433312,0.514708
4,0.433312,-0.545285


## 4. Top 7 Key Insights
Based on our entire process of Data Cleaning, EDA, and Feature Engineering, here are 7 of the most interesting findings:

1. **Gender Significantly Impacted Survival**: The survival rate for females was overwhelmingly higher than for males. Women and children were prioritized during the evacuation.
2. **Passenger Class Mattered**: Survival rates were significantly higher for 1st Class passengers compared to 3rd Class passengers, reflecting socioeconomic impacts during the disaster.
3. **Age Distribution**: Most passengers were young adults between 20 and 35. Younger individuals and children generally had better survival odds.
4. **Most Passengers Traveled Alone**: By engineering `FamilySize` and `IsAlone`, we found the majority traveled solo. This demographic (largely adult males) experienced the highest casualty rates.
5. **Cabin Data Correlation**: Engineering `HasCabin` proved highly useful. Although `Cabin` data was largely missing, the mere recording of a cabin number correlated strongly with 1st-class tickets and much higher survival probabilities.

6. **Data Scaling Equalizes Impact**: By applying a `StandardScaler` to `Age` and `Fare`, we converted them into Z-scores (centered around 0). This ensures that extreme ticket prices (Fares) won't unfairly dominate machine learning algorithms just because their raw numbers are larger than a person's age.